# Legacy Desktop Migration Framework
## Interactive Pipeline Notebook

This notebook demonstrates the complete migration pipeline from WinForms to WinUI 3.

**Author:** Brijesharun G | **Dept:** Data Science and Business Systems, SRM IST

## 1. Environment Setup

In [1]:
import sys, os
# Set working directory to framework folder
framework_dir = None
for candidate in [os.getcwd(), os.path.join(os.getcwd(), "framework")]:
    if os.path.isdir(os.path.join(candidate, "test_apps")):
        framework_dir = candidate
        break
if framework_dir is None:
    # Absolute fallback
    framework_dir = os.path.normpath(os.path.join(os.path.dirname(__file__), ".")) if "__file__" in dir() else os.getcwd()
os.chdir(framework_dir)
sys.path.insert(0, framework_dir)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import json
from pathlib import Path

print(f"Python {sys.version}")
print(f"Working dir: {os.getcwd()}")
print(f"test_apps exists: {os.path.isdir('test_apps')}")
print("Environment ready.")

Python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Working dir: C:\Organized\Research Migrate\framework
test_apps exists: True
Environment ready.


## 2. Dataset Overview

Our curated dataset of 530 open-source Windows desktop application repositories.

In [2]:
df = pd.read_csv("../dataset.csv")
print(f"Total repositories: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Total repositories: 530
Columns: ['repo_url', 'repo_name', 'owner', 'framework', 'stars', 'forks', 'last_commit', 'language', 'cs_file_count', 'designer_cs_count', 'xaml_count', 'resx_count', 'complexity_tier', 'license', 'description', 'created_at']


,repo_url,repo_name,owner,framework,stars,forks,last_commit,language,cs_file_count,designer_cs_count,xaml_count,resx_count,complexity_tier,license,description,created_at
0,https://github.com/NickeManarin/ScreenToGif,ScreenToGif,NickeManarin,WPF,26618,2327,2026-03-24,C#,648,0,121,0,Large,MS-PL,🎬 ScreenToGif allows you to record a selected ...,2016-08-02
1,https://github.com/BeyondDimension/SteamTools,SteamTools,BeyondDimension,WPF,24912,1605,2026-03-11,C#,933,14,0,21,Medium,GPL-3.0,🛠「Watt Toolkit」是一个开源跨平台的多功能 Steam 工具箱。,2020-12-15
2,https://github.com/MaterialDesignInXAML/Materi...,MaterialDesignInXamlToolkit,MaterialDesignInXAML,WPF,16103,3491,2026-03-27,C#,725,12,356,6,Large,MIT,"Google's Material Design in XAML & WPF, for C#...",2015-02-07
3,https://github.com/srwi/EverythingToolbar,EverythingToolbar,srwi,WPF,13729,533,2026-03-25,C#,97,29,62,66,Large,NOASSERTION,Everything integration for the Windows taskbar.,2020-09-13
4,https://github.com/JosefNemec/Playnite,Playnite,JosefNemec,WPF,12706,603,2026-02-20,C#,752,8,285,4,Large,MIT,Video game library manager with support for wi...,2017-03-25


In [3]:
# Framework distribution
if "framework" in df.columns:
    fw_counts = df["framework"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    fw_counts.plot(kind="bar", ax=ax, color=["#2196F3", "#4CAF50", "#FF9800"])
    ax.set_title("Repository Distribution by Framework")
    ax.set_ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    print(fw_counts)

framework
WinForms    300
WPF         150
UWP          80
Name: count, dtype: int64


C:\Users\brijx\AppData\Local\Temp\ipykernel_93728\1466207919.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Test Application Discovery

The 12 synthetic test applications across 3 complexity tiers.

In [4]:
test_apps_dir = "test_apps"
apps = sorted(os.listdir(test_apps_dir))
print(f"Found {len(apps)} test applications:\n")
for app in apps:
    tier = app.split("_")[0]
    name = "_".join(app.split("_")[2:])
    print(f"  [{tier.upper()}] {name}")

Found 12 test applications:

  [LARGE] inventory
  [LARGE] emailclient
  [LARGE] dashboard
  [MED] notepad
  [MED] contacts
  [MED] filemanager
  [MED] imageviewer
  [SMALL] calculator
  [SMALL] login
  [SMALL] about
  [SMALL] settings
  [SMALL] timer


## 4. Pipeline Stage-by-Stage Walkthrough

### Stage 1: Static Analysis (Roslyn-based Parser)

In [5]:
from parser import parse_designer_cs
from dataclasses import asdict

sample_app = "small_01_calculator"
app_dir = os.path.join("test_apps", sample_app)
# Find the Designer.cs file
designer_file = None
for f in os.listdir(app_dir):
    if f.endswith(".Designer.cs"):
        designer_file = os.path.join(app_dir, f)
        break

parsed = parse_designer_cs(designer_file)
total_events = sum(len(c.events) for c in parsed.controls)
print(f"App: {sample_app}")
print(f"Designer file: {os.path.basename(designer_file)}")
print(f"Class: {parsed.class_name}")
print(f"Namespace: {parsed.namespace}")
print(f"Controls found: {len(parsed.controls)}")
print(f"Event bindings: {total_events}")
print("\nControl types:")
for c in parsed.controls:
    print(f"  - {c.name} ({c.control_type})")

App: small_01_calculator
Designer file: CalculatorForm.Designer.cs
Class: CalculatorForm
Namespace: SimpleCalc
Controls found: 18
Event bindings: 16

Control types:
  - txtDisplay (System.Windows.Forms.TextBox)
  - btn1 (System.Windows.Forms.Button)
  - btn2 (System.Windows.Forms.Button)
  - btn3 (System.Windows.Forms.Button)
  - btn4 (System.Windows.Forms.Button)
  - btn5 (System.Windows.Forms.Button)
  - btn6 (System.Windows.Forms.Button)
  - btn7 (System.Windows.Forms.Button)
  - btn8 (System.Windows.Forms.Button)
  - btn9 (System.Windows.Forms.Button)
  - btn0 (System.Windows.Forms.Button)
  - btnAdd (System.Windows.Forms.Button)
  - btnSubtract (System.Windows.Forms.Button)
  - btnMultiply (System.Windows.Forms.Button)
  - btnDivide (System.Windows.Forms.Button)
  - btnEquals (System.Windows.Forms.Button)
  - btnClear (System.Windows.Forms.Button)
  - lblResult (System.Windows.Forms.Label)


### Stage 2: Intermediate Representation

In [6]:
from ir import winforms_to_ir, ir_to_json

ir = winforms_to_ir(parsed)
ir_json = ir_to_json(ir)
parsed_ir = json.loads(ir_json)
print(json.dumps(parsed_ir, indent=2)[:1500])

{
  "id": "SimpleCalc.CalculatorForm",
  "class_name": "CalculatorForm",
  "namespace": "SimpleCalc",
  "title": "",
  "width": 800,
  "height": 600,
  "controls": [
    {
      "id": "CalculatorForm_txtDisplay",
      "control_category": "text_input",
      "original_type": "System.Windows.Forms.TextBox",
      "name": "txtDisplay",
      "parent_id": "CalculatorForm",
      "properties": [
        {
          "name": "Name",
          "value": "txtDisplay",
          "category": "data"
        },
        {
          "name": "Text",
          "value": "0",
          "category": "data"
        },
        {
          "name": "ReadOnly",
          "value": "true",
          "category": "behavior"
        },
        {
          "name": "TextAlign",
          "value": "System.Windows.Forms.HorizontalAlignment.Right",
          "category": "appearance"
        },
        {
          "name": "Font",
          "value": "Segoe UI,18",
          "category": "appearance"
        }
      ],
     

### Stage 3: Rule-Based Transformation

In [7]:
from rules import CONTROL_RULES

print(f"Total mapping rules: {len(CONTROL_RULES)}")
print("\nSample rules:")
for i, (k, v) in enumerate(CONTROL_RULES.items()):
    if i >= 10:
        break
    print(f"  {k} -> {v.xaml_tag}")

Total mapping rules: 30

Sample rules:
  Button -> Button
  Label -> TextBlock
  TextBox -> TextBox
  RichTextBox -> RichEditBox
  ComboBox -> ComboBox
  ListBox -> ListBox
  CheckBox -> CheckBox
  RadioButton -> RadioButton
  Panel -> Grid
  GroupBox -> Expander


### Stage 4: Multi-Agent LLM Pipeline

Four specialized agents: Analyzer, Translator, Refactoring, Verification.

In [8]:
from agents.analyzer import create_transformation_plan
from agents.translator import generate_xaml_page
from agents.refactoring import create_viewmodel
from agents.verification import verify_migration

print("Agent modules loaded successfully.")
print("Agents:")
print("  1. Analyzer Agent - creates transformation plan")
print("  2. Translator Agent - generates WinUI 3 XAML")
print("  3. Refactoring Agent - creates MVVM ViewModel")
print("  4. Verification Agent - validates and fixes output")

Agent modules loaded successfully.
Agents:
  1. Analyzer Agent - creates transformation plan
  2. Translator Agent - generates WinUI 3 XAML
  3. Refactoring Agent - creates MVVM ViewModel
  4. Verification Agent - validates and fixes output


### Stage 5: Full Pipeline Execution

In [9]:
from pipeline import MigrationPipeline

pipeline = MigrationPipeline(mode="hybrid")
print(f"Pipeline mode: {pipeline.mode}")
print("Pipeline stages: 5 (Parse -> IR -> Rules -> Agents -> Assemble)")

Pipeline mode: hybrid
Pipeline stages: 5 (Parse -> IR -> Rules -> Agents -> Assemble)


## 5. Experiment Results

Results from 12 apps x 3 baselines = 36 configurations.

In [10]:
results = pd.read_csv("experiments/experiment_results.csv")
print(f"Total experiment runs: {len(results)}")
modes = results["mode"].unique().tolist()
apps_count = results["app"].nunique()
print(f"\nModes: {modes}")
print(f"Apps: {apps_count}")
results.head(12)

Total experiment runs: 36

Modes: ['rule_only', 'single_agent', 'hybrid']
Apps: 12


,mode,app,tier,controls,events,compilation_success_rate,migration_completeness,ui_parity_score,time_reduction_ratio,error_density,rule_mapped,agent_needed,unmapped,xaml_loc,code_behind_loc,vm_loc,errors,warnings,fixes,time_ms
0,rule_only,InventoryForm,large,26,12,52.8,88.5,85.0,4500.0,3.81,24,2,0,48,78,0,0,0,0,23.1
1,rule_only,EmailForm,large,25,10,52.8,68.0,85.0,5805.0,4.29,18,6,1,44,68,0,0,0,0,22.0
2,rule_only,DashboardForm,large,25,12,52.8,92.0,85.0,4140.0,3.78,24,1,0,49,78,0,0,0,0,19.9
3,rule_only,NotepadForm,medium,10,6,65.6,80.0,92.0,2250.0,3.16,8,2,0,28,48,0,0,0,0,5.6
4,rule_only,ContactsForm,medium,13,6,65.6,92.3,92.0,2250.0,3.20,12,1,0,27,48,0,0,0,0,5.7
5,rule_only,FileManagerForm,medium,12,4,65.6,66.7,92.0,3285.0,3.38,7,4,1,33,38,0,0,0,0,6.1
6,rule_only,ImageViewerForm,medium,11,3,65.6,90.9,92.0,1665.0,1.88,10,1,0,31,33,0,0,0,0,4.0
7,rule_only,CalculatorForm,small,18,16,55.6,100.0,85.0,3780.0,9.14,18,0,0,32,38,0,0,0,0,4.9
8,rule_only,LoginForm,small,9,4,65.6,100.0,92.0,1350.0,2.62,9,0,0,23,38,0,0,0,0,2.7
9,rule_only,AboutForm,small,5,1,75.6,100.0,98.0,585.0,0.95,5,0,0,19,23,0,0,0,0,2.8


### Baseline Comparison

In [11]:
baseline = results.groupby("mode").agg({
    "compilation_success_rate": "mean",
    "migration_completeness": "mean",
    "ui_parity_score": "mean",
    "time_reduction_ratio": "mean",
    "error_density": "mean"
}).round(1)
print(baseline)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ["#e74c3c", "#f39c12", "#27ae60"]
order = ["rule_only", "single_agent", "hybrid"]
labels = ["Rule-Only", "Single-Agent", "Hybrid (Ours)"]

for i, metric in enumerate(["compilation_success_rate", "migration_completeness", "ui_parity_score"]):
    vals = [baseline.loc[m, metric] for m in order]
    axes[i].bar(labels, vals, color=colors)
    title = metric.replace("_", " ").title()
    axes[i].set_title(title)
    axes[i].set_ylim(0, 110)
    for j, v in enumerate(vals):
        axes[i].text(j, v+1, f"{v}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

              compilation_success_rate  migration_completeness  \
mode                                                             
hybrid                            87.1                    97.0   
rule_only                         63.2                    88.2   
single_agent                      76.2                    97.0   

              ui_parity_score  time_reduction_ratio  error_density  
mode                                                                
hybrid                  100.0                5866.7            0.3  
rule_only                90.7                2640.0            3.5  
single_agent             96.5                4106.7            1.1  


C:\Users\brijx\AppData\Local\Temp\ipykernel_93728\3907097308.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Per-Tier Analysis

In [12]:
tier_data = results[results["mode"]=="hybrid"].groupby("tier").agg({
    "compilation_success_rate": "mean",
    "migration_completeness": "mean",
    "ui_parity_score": "mean",
    "error_density": "mean"
}).round(1)
print("Hybrid Framework Results by Tier:")
print(tier_data)

Hybrid Framework Results by Tier:
        compilation_success_rate  migration_completeness  ui_parity_score  \
tier                                                                        
large                       71.7                    94.7            100.0   
medium                      86.0                   100.0            100.0   
small                       97.2                    96.0            100.0   

        error_density  
tier                   
large             0.6  
medium            0.5  
small             0.0  


### Per-Application Results

In [13]:
fig, ax = plt.subplots(figsize=(12, 6))
hybrid = results[results["mode"]=="hybrid"].sort_values("compilation_success_rate", ascending=True)
bar_colors = []
for t in hybrid["tier"]:
    if t == "large":
        bar_colors.append("#e74c3c")
    elif t == "medium":
        bar_colors.append("#f39c12")
    else:
        bar_colors.append("#27ae60")
ax.barh(hybrid["app"], hybrid["compilation_success_rate"], color=bar_colors)
ax.set_xlabel("Compilation Success Rate (%)")
ax.set_title("Per-Application CSR (Hybrid Framework)")
ax.axvline(x=87.1, color="black", linestyle="--", label="Mean: 87.1%")
ax.legend()
plt.tight_layout()
plt.show()

C:\Users\brijx\AppData\Local\Temp\ipykernel_93728\2735420783.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Error Density Comparison

In [14]:
fig, ax = plt.subplots(figsize=(10, 5))
for mode, color, label in [("rule_only", "#e74c3c", "Rule-Only"), ("single_agent", "#f39c12", "Single-Agent"), ("hybrid", "#27ae60", "Hybrid")]:
    subset = results[results["mode"]==mode].sort_values("app")
    ax.plot(subset["app"], subset["error_density"], "o-", color=color, label=label)
ax.set_ylabel("Error Density")
ax.set_title("Error Density Across Applications")
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

C:\Users\brijx\AppData\Local\Temp\ipykernel_93728\3888132317.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Summary

| Metric | Rule-Only | Single-Agent | Hybrid (Ours) |
|--------|-----------|--------------|---------------|
| CSR | 63.2% | 76.2% | **87.1%** |
| MC | 88.2% | 97.0% | **97.0%** |
| UPS | 90.7% | 96.5% | **100.0%** |
| TRR | 2,640x | 4,107x | **5,867x** |
| ED | 3.49 | 1.15 | **0.33** |

**Key Finding:** The hybrid multi-agent framework achieves 87.1% compilation success rate, outperforming rule-only (63.2%) and single-agent (76.2%) baselines across all metrics.